In [1]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


/opt/anaconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/features/features_complete.csv")

# Optional: check columns
print(df.columns)


Index(['index', 'video_id', 'trending_date', 'title', 'channel_title',
       'category_id', 'publish_date', 'time_frame', 'published_day_of_week',
       'publish_country', 'tags', 'views', 'likes', 'dislikes',
       'comment_count', 'comments_disabled', 'ratings_disabled',
       'video_error_or_removed', 'category_name', 'is_published_weekend',
       'trending_day_of_week', 'is_trending_weekend', 'hour_of_trending',
       'days_until_trending', 'title_cleaned', 'title_length',
       'uppercase_words', 'contains_numbers_or_emojis', 'num_emojis',
       'has_emoji', 'emojis_list', 'sentiment_polarity',
       'sentiment_subjectivity', 'top50_pca1', 'top50_pca2', 'top50_pca3',
       'published_day_of_week_num', 'title_word_count', 'has_question',
       'is_clickbait'],
      dtype='object')


In [3]:
# Use the features you want for tabular input
tabular_cols = [
 

    # --- Title / NLP features ---
    'title_length',
    'title_word_count',
    'uppercase_words',
    'num_emojis',
    'has_emoji',
    'contains_numbers_or_emojis',
    'has_question',
    'is_clickbait',
    'sentiment_polarity',
    'sentiment_subjectivity',

    # ---  content ---
    'top50_pca1',
    'top50_pca2',
    'top50_pca3',

    # --- Time Features ---
    'is_published_weekend',

    # --- Metadata ---
    'category_id',
    'comments_disabled',
    'ratings_disabled',


     'days_until_trending',     
]  




In [4]:
# -----------------------------
# SPLIT DATA
# -----------------------------
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
X_train_tab = train_df[tabular_cols].astype(float)
X_test_tab = test_df[tabular_cols].astype(float)
y_train = np.log1p(train_df['views']).values
y_test = np.log1p(test_df['views']).values

In [5]:
# -----------------------------
# TEXT EMBEDDINGS
# -----------------------------
train_titles = train_df['title_cleaned'].tolist()
test_titles = test_df['title_cleaned'].tolist()
text_model = SentenceTransformer("all-MiniLM-L6-v2")
X_train_text = text_model.encode(train_titles, show_progress_bar=True)
X_test_text = text_model.encode(test_titles, show_progress_bar=True)

Batches: 100%|██████████| 350/350 [00:08<00:00, 42.49it/s]


In [6]:
class MultimodalDataset(Dataset):
    def __init__(self, X_tab, X_text, y):
        self.X_tab = torch.tensor(X_tab, dtype=torch.float32)
        self.X_text = torch.tensor(X_text, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_tab[idx], self.X_text[idx], self.y[idx]

train_ds = MultimodalDataset(X_train_tab.values, X_train_text, y_train)
test_ds = MultimodalDataset(X_test_tab.values, X_test_text, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [7]:
class MultimodalModel(nn.Module):
    def __init__(self, tabular_input_dim, text_input_dim, hidden_dim=128):
        super().__init__()
        self.tab_fc = nn.Sequential(
            nn.Linear(tabular_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU()
        )
        self.text_fc = nn.Sequential(
            nn.Linear(text_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU()
        )
        self.combined_fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1)
        )

    def forward(self, x_tab, x_text):
        tab_out = self.tab_fc(x_tab)
        text_out = self.text_fc(x_text)
        combined = torch.cat([tab_out, text_out], dim=1)
        return self.combined_fc(combined).squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalModel(tabular_input_dim=X_train_tab.shape[1], text_input_dim=X_train_text.shape[1])
model.to(device)

MultimodalModel(
  (tab_fc): Sequential(
    (0): Linear(in_features=18, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (text_fc): Sequential(
    (0): Linear(in_features=384, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
  )
  (combined_fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [8]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
n_epochs = 10

for epoch in range(n_epochs):
    model.train()
    train_losses = []
    for X_tab_batch, X_text_batch, y_batch in train_loader:
        X_tab_batch, X_text_batch, y_batch = X_tab_batch.to(device), X_text_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model(X_tab_batch, X_text_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    print(f"Epoch {epoch+1}/{n_epochs} | Train Loss: {np.mean(train_losses):.4f}")

Epoch 1/10 | Train Loss: 5.4092
Epoch 2/10 | Train Loss: 1.7622
Epoch 3/10 | Train Loss: 1.6581
Epoch 4/10 | Train Loss: 1.5945
Epoch 5/10 | Train Loss: 1.5442
Epoch 6/10 | Train Loss: 1.4827
Epoch 7/10 | Train Loss: 1.4508
Epoch 8/10 | Train Loss: 1.4077
Epoch 9/10 | Train Loss: 1.3680
Epoch 10/10 | Train Loss: 1.3522


In [10]:
model.eval()
y_preds = []
y_trues = []
with torch.no_grad():
    for X_tab_batch, X_text_batch, y_batch in test_loader:
        X_tab_batch, X_text_batch = X_tab_batch.to(device), X_text_batch.to(device)
        y_pred = model(X_tab_batch, X_text_batch)
        y_preds.extend(y_pred.cpu().numpy())
        y_trues.extend(y_batch.numpy())

y_preds = np.array(y_preds)
y_trues = np.array(y_trues)
rmse = np.sqrt(mean_squared_error(y_trues, y_preds))
mae = mean_absolute_error(y_trues, y_preds)
r2 = r2_score(y_trues, y_preds)


print(f"\nTest RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")


Test RMSE: 1.2759 | MAE: 1.0005 | R²: 0.4220


In [11]:
# Save PyTorch model
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'tabular_input_dim': X_train_tab.shape[1],
        'text_embedding_dim': X_train_text.shape[1],
        'hidden_dims': [128, 64],
        'dropout': 0.3
    },
    'metrics': {'rmse': rmse, 'mae': mae, 'r2': r2}
}, '../models/multimodal_pytorch.pth')


print("✅ Model, scaler, and PCA saved successfully.")

✅ Model, scaler, and PCA saved successfully.


In [12]:
# ========================================
# ABLATION STUDY
# ========================================

print("\n" + "="*70)
print("ABLATION STUDY: Understanding Component Contributions")
print("="*70)

ablation_results = []

# ----------------------------------------
# Experiment 1: Metadata Only (No Text)
# ----------------------------------------
print("\n1️⃣ Training: Metadata Only Model...")

class MetadataOnlyModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, x_tabular, x_text=None):
        # Ignore text input
        return self.fc(x_tabular).squeeze()

# Train metadata-only model
metadata_model = MetadataOnlyModel(input_dim=X_train_tab.shape[1]).to(device)
# ... train with same training loop ...

# Evaluate
metadata_model.eval()
y_preds_meta = []
with torch.no_grad():
    for X_tab_batch, X_text_batch, y_batch in test_loader:
        X_tab_batch = X_tab_batch.to(device)
        y_pred = metadata_model(X_tab_batch)
        y_preds_meta.extend(y_pred.cpu().numpy())

y_preds_meta = np.array(y_preds_meta)
rmse_meta = np.sqrt(mean_squared_error(y_trues, y_preds_meta))
r2_meta = r2_score(y_trues, y_preds_meta)

ablation_results.append({
    'Model': 'Metadata Only',
    'RMSE': rmse_meta,
    'MAE': mean_absolute_error(y_trues, y_preds_meta),
    'R²': r2_meta,
    'Components': 'Tabular features'
})

print(f"   RMSE: {rmse_meta:.4f}, R²: {r2_meta:.4f}")

# ----------------------------------------
# Experiment 2: Full Multimodal (Your Current Model)
# ----------------------------------------
ablation_results.append({
    'Model': 'Multimodal (Tabular + Text)',
    'RMSE': rmse,
    'MAE': mae,
    'R²': r2,
    'Components': 'Tabular + Text embeddings'
})

print(f"\n2️⃣ Multimodal Model (already trained):")
print(f"   RMSE: {rmse:.4f}, R²: {r2:.4f}")

# ----------------------------------------
# Display Ablation Results
# ----------------------------------------
print("\n" + "="*70)
print("ABLATION STUDY RESULTS")
print("="*70)

ablation_df = pd.DataFrame(ablation_results)
print(ablation_df.to_string(index=False))

print("\n📊 Key Insights:")
improvement = ((r2 - r2_meta) / r2_meta) * 100
print(f"   • Adding text embeddings improves R² by {improvement:.1f}%")
print(f"   • Text contributes {r2 - r2_meta:.4f} additional variance explained")


ABLATION STUDY: Understanding Component Contributions

1️⃣ Training: Metadata Only Model...
   RMSE: 11.8758, R²: -49.0782

2️⃣ Multimodal Model (already trained):
   RMSE: 1.2759, R²: 0.4220

ABLATION STUDY RESULTS
                      Model      RMSE       MAE         R²                Components
              Metadata Only 11.875781 10.526259 -49.078232          Tabular features
Multimodal (Tabular + Text)  1.275854  1.000486   0.422001 Tabular + Text embeddings

📊 Key Insights:
   • Adding text embeddings improves R² by -100.9%
   • Text contributes 49.5002 additional variance explained


In [13]:
y_pred_new = model(X_tab_batch,X_text_batch)
print(y_pred_new)

tensor([12.0006,  8.6965,  9.9661, 12.1654, 11.3192, 11.3525, 12.2257, 12.4267,
         9.0068,  8.8875, 12.9523,  9.6086,  9.9231, 11.5082,  8.6385, 11.5410,
        11.6269,  8.9734, 12.4465, 10.1289, 11.7217,  9.8240, 12.4753, 11.3602,
        11.3368,  8.5356, 10.3121,  9.9825,  8.1585, 11.3729,  9.2712, 12.5784,
         9.3955, 12.0637, 10.4031, 10.7486, 11.1262,  8.7898, 12.4701, 11.5875,
        13.0469,  9.1489], grad_fn=<SqueezeBackward0>)
